# Carga de Telco Clean → PostgreSQL

## Esquema de tablas

```
dim_estado        (id, descripcion)  → 'No'(0), 'Sí'(1), 'Sin servicio'(2)
dim_genero        (id, descripcion)  → 'Femenino'(0), 'Masculino'(1)
dim_multiple_lines(id, descripcion)  → 'No'(0), 'Sí'(1), 'Sin servicio telefónico'(2)
dim_internet      (id, descripcion)  → 'No'(0), 'DSL'(1), 'Fiber optic'(2)
dim_contrato      (id, descripcion)  → 'Month-to-month'(0), 'One year'(1), 'Two year'(2)
dim_metodo_pago   (id, descripcion)  → 4 métodos de pago
cliente           (PK + FKs + campos numéricos)
```

> ⚠️ Las tablas y catálogos ya están creados por `telco_schema.sql`  
> vía `docker-entrypoint-initdb.d`. Este notebook **solo carga** la tabla `cliente`.

## Requisitos
```
pip install pandas sqlalchemy psycopg2-binary python-dotenv
```

## Archivo .env requerido
Crea un archivo `.env` en la raíz del repo con:
```
DB_HOST=localhost
DB_PORT=5432
DB_NAME=telco_db
DB_USER=telco_user
DB_PASSWORD=telco_pass
```

In [ ]:
%pip install pandas sqlalchemy psycopg2-binary python-dotenv

## 1. Importar librerías

In [ ]:
import pandas as pd
import os
import logging
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

print("Librerías cargadas correctamente")

## 2. Configurar logger

In [ ]:
os.makedirs("logs", exist_ok=True)

logging.basicConfig(
    filename="logs/carga_telco.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

logging.info("Inicio del proceso de carga Telco → PostgreSQL")
print("Logger configurado")

## 3. Conectar a PostgreSQL

In [ ]:
load_dotenv(".env")

DB_HOST     = os.getenv("DB_HOST",     "localhost")
DB_PORT     = os.getenv("DB_PORT",     "5432")
DB_NAME     = os.getenv("DB_NAME",     "telco_db")
DB_USER     = os.getenv("DB_USER",     "telco_user")
DB_PASSWORD = os.getenv("DB_PASSWORD", "telco_pass")

connection_string = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(connection_string)

try:
    with engine.connect() as conn:
        result = conn.execute(text("SELECT current_database(), current_schema();"))
        db, schema = result.fetchone()
        logging.info(f"Conexión exitosa → DB: {db}, Schema: {schema}")
        print(f"Conectado a: base={db}, schema={schema}")
except Exception as e:
    logging.error(f"Error de conexión: {e}")
    print(f"Error de conexión: {e}")
    raise

## 4. Verificar que las tablas del schema existen

> Las tablas fueron creadas automáticamente por `telco_schema.sql` al iniciar el contenedor.  
> Esta celda solo confirma que están disponibles antes de cargar datos.

In [ ]:
tablas_esperadas = [
    "dim_estado", "dim_genero", "dim_multiple_lines",
    "dim_internet", "dim_contrato", "dim_metodo_pago", "cliente"
]

try:
    with engine.connect() as conn:
        result = conn.execute(text("""
            SELECT table_name
            FROM information_schema.tables
            WHERE table_schema = 'public'
            ORDER BY table_name;
        """))
        tablas_db = [row[0] for row in result]

    faltantes = [t for t in tablas_esperadas if t not in tablas_db]

    if faltantes:
        raise RuntimeError(f"Tablas faltantes en la DB: {faltantes}. Verifica que telco_schema.sql se ejecutó correctamente.")
    else:
        print("✅ Todas las tablas encontradas:")
        for t in tablas_db:
            print(f"   - {t}")
        logging.info("Verificación de tablas OK")
except Exception as e:
    logging.error(f"Error verificando tablas: {e}")
    print(f"Error: {e}")
    raise

## 5. Leer y preparar el CSV limpio

In [ ]:
df = pd.read_csv('../data/processed/Teleco_validado.csv')
print(f"Filas leídas: {len(df)}")
print(f"Columnas: {df.columns.tolist()}")
df.head(3)

In [ ]:
# Renombrar columnas al esquema de la tabla cliente
df_cliente = df.rename(columns={
    "gender":            "genero_id",
    "multiple_lines":    "multiple_lines_id",
    "internet_service":  "internet_id",
    "online_security":   "online_security_id",
    "online_backup":     "online_backup_id",
    "device_protection": "device_protection_id",
    "tech_support":      "tech_support_id",
    "streaming_tv":      "streaming_tv_id",
    "streaming_movies":  "streaming_movies_id",
    "contract":          "contrato_id",
    "payment_method":    "metodo_pago_id",
})

# Convertir columnas booleanas (0/1 → True/False)
bool_cols = ["senior_citizen", "partner", "dependents",
             "phone_service", "paperless_billing", "churn"]
for col in bool_cols:
    df_cliente[col] = df_cliente[col].astype(bool)

print("Columnas finales del DataFrame:")
print(df_cliente.dtypes)
df_cliente.head(3)

## 6. Cargar tabla cliente

In [ ]:
try:
    filas_insertadas = df_cliente.to_sql(
        name="cliente",
        con=engine,
        if_exists="append",  # 'replace' borra y recrea; 'append' agrega
        index=False,
        method="multi",      # inserciones en lote (más rápido)
        chunksize=500
    )
    logging.info(f"Tabla cliente cargada: {filas_insertadas} filas")
    print(f"Tabla cliente cargada: {filas_insertadas} filas insertadas")
except Exception as e:
    logging.error(f"Error cargando tabla cliente: {e}")
    print(f"Error: {e}")
    raise

## 7. Verificar carga

In [ ]:
queries_verificacion = {
    "Total clientes":     "SELECT COUNT(*) FROM cliente;",
    "Clientes con churn": "SELECT COUNT(*) FROM cliente WHERE churn = TRUE;",
    "Tipos de contrato": """
        SELECT c.descripcion, COUNT(*) AS total
        FROM cliente cl
        JOIN dim_contrato c ON cl.contrato_id = c.id
        GROUP BY c.descripcion ORDER BY total DESC;
    """,
    "Servicio de internet": """
        SELECT i.descripcion, COUNT(*) AS total
        FROM cliente cl
        JOIN dim_internet i ON cl.internet_id = i.id
        GROUP BY i.descripcion ORDER BY total DESC;
    """,
    "Vista v_clientes (muestra)": """
        SELECT customer_id, genero, contrato, internet_service, churn
        FROM v_clientes
        LIMIT 5;
    """
}

with engine.connect() as conn:
    for nombre, sql in queries_verificacion.items():
        result = pd.read_sql(text(sql), conn)
        print(f"\n=== {nombre} ===")
        print(result.to_string(index=False))

logging.info("Verificación completada")
print("\n✅ Carga completada exitosamente")